# 55. Jump Game
**Difficulty:** 🟡 Medium · **Topic:** Dynamic Programming · **LeetCode:** https://leetcode.com/problems/jump-game/

## 💡 Concepts

**Core concept(s):** DP over reachability, but a **greedy** scan does it in one pass.

**Why it applies here:** Each value is the max jump from that index. DP asks, for each cell, whether some earlier reachable cell can jump to it (`O(n²)`). The greedy insight: sweep once tracking the farthest index reachable so far; if you ever can't advance to the current index, you're stuck.

**Key intuition:** Track the farthest you can reach; if the current index is beyond it, you can't get there.

---

### 📚 What is Dynamic Programming (DP)?
**DP** solves a big problem by solving smaller **overlapping** subproblems once and reusing the answers. Two styles: **memoization** (recursion that caches results) and **tabulation** (fill a table from the smallest cases up).
- **Why it's fast:** it turns exponential re-computation into a single sweep over the subproblems.
- **In Python:** a `dict`/list cache, or a `dp` list/2-D table.

### 📚 What is Greedy?
**Greedy** makes the best local choice and never looks back. It only works when local choices provably build a global optimum — then it beats DP on speed and space.

---

**Prerequisite knowledge:**
- Reachability DP.
- The greedy 'farthest reach' scan.

## 📝 Problem

Each `nums[i]` is the maximum jump length from index `i`. Starting at index 0, can you reach the last index?

**Example**
```
[2,3,1,1,4] -> True
[3,2,1,0,4] -> False   (stuck at the 0)
```

> Two approaches: DP `O(n²)` and greedy `O(n)`.

### Approach 1 — Reachability DP (worst)

**Idea:** `dp[i]` = can we reach index `i`? True if some reachable earlier index can jump to it.

**Time:** `O(n²)`. **Space:** `O(n)`.

In [ ]:
def can_jump_dp(nums):
    n = len(nums)
    dp = [False] * n                       # dp[i] = can we reach index i?
    dp[0] = True                           # we start at index 0
    for i in range(n):
        if not dp[i]:
            continue                       # can't reach i, so it can't launch us anywhere
        for j in range(1, nums[i] + 1):    # from i we can land on any of the next nums[i] cells
            if i + j < n:
                dp[i + j] = True
    return dp[n - 1]

### Approach 2 — Greedy Farthest Reach (optimal)

**Idea:** Sweep left to right, keeping the farthest index reachable. If the current index exceeds it, you can never get here → fail.

**Time:** `O(n)`. **Space:** `O(1)`.

In [ ]:
def can_jump_greedy(nums):
    farthest = 0                           # the farthest index we can currently reach
    for i in range(len(nums)):
        if i > farthest:
            return False                   # this index is beyond our reach -> stuck
        farthest = max(farthest, i + nums[i])   # extend our reach from here
    return True                            # never fell behind -> the end is reachable

In [ ]:
# Correctness check
tests = [([2,3,1,1,4],True), ([3,2,1,0,4],False), ([0],True), ([2,0,0],True), ([1,0,1],False)]
for nums, exp in tests:
    a, b = can_jump_dp(nums), can_jump_greedy(nums)
    print(f"{nums} -> dp={a}, greedy={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |

Inputs are shaped to force the worst case. (Exponential brute-force versions are shown in the code but omitted from timing where they would blow up — noted per notebook.)

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    # every index can jump far -> the DP fills many cells (near O(n^2)), greedy stays O(n)
    return ([n] * n,)
solutions = {
    "dp     O(n^2)": can_jump_dp,
    "greedy O(n)  ": can_jump_greedy,
}
sizes = [500, 1000, 2000, 4000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Greedy can beat DP:** tracking the farthest reach replaces an `O(n²)` reachability table with an `O(n)` scan.
- **Reachability sweep:** carry a running 'best reach' and check you never fall behind.
- **Signal:** "can you reach the end / minimum jumps", "coverage as you scan".
- **Related problems:** Jump Game II (min jumps), Gas Station, Video Stitching.
- **Common pitfalls:** (1) defaulting to O(n²) DP when greedy suffices; (2) off-by-one on the reach comparison.